In [4]:
!pip install optuna

In [5]:
!pip install lightgbm

#### Importing required libraries 

In [1]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import *
import pandas as pd
import time
import optuna
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")

In [2]:
file_path_replies = r"../replies_sydneysiege.pkl"
file_path_posts = r"../posts_sydneysiege.pkl"

In [3]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=3*60*24)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [4]:
train.head()

,followers,favorite_count,retweet_count,first_time_diff,replies,no_verified,verified,embeddings_avg,rumour,min_since_fst_post
0,-0.129424,-0.252174,1.517832,1.515602,0.375,0,1,"[0.04698175168596208, -0.18934187246486545, -0...",1,5.62
1,-0.128771,-0.617391,-0.171184,3.964339,-0.500,0,1,"[0.1690062526613474, -0.11465575313195586, 0.1...",1,7.73
2,-0.092106,-0.382609,0.131241,-0.023774,0.750,0,1,"[-0.23375208879059012, -0.19412890686230225, -...",1,9.52
3,0.091430,-0.643478,-0.405136,-0.380386,-0.500,0,1,"[0.029154916604359944, -0.25932883098721504, -...",1,9.90
4,-0.129424,0.008696,2.773181,1.384844,0.625,0,1,"[-0.03537977912596294, -0.043058500226054876, ...",1,11.85


In [5]:
previous_node_count = 0

In [6]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 352


#### Tunning Random Forest

In [124]:


n_train = len(X_train)  
def objective_rf(trial, X, y):
    # sensible RF hyperparameter search space
    param_grid = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 5),                     # tree depth
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),     # internal node split
        # min_samples_leaf relative to dataset size but never larger than n_train:
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            max(2, min(n_train, int(max(2, n_train * 0.2))))
        ),
        # max_features: either a rule or a fraction; use categorical choices for stability
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        # optional class weight (useful for imbalance)
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
    }

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1337)
    cv_scores = np.empty(cv.get_n_splits())

    for idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = RandomForestClassifier(
            n_estimators=param_grid["n_estimators"],
            max_depth=param_grid["max_depth"],
            min_samples_split=param_grid["min_samples_split"],
            min_samples_leaf=param_grid["min_samples_leaf"],
            max_features=param_grid["max_features"],
            bootstrap=param_grid["bootstrap"],
            class_weight=param_grid["class_weight"],
            n_jobs=-1,
            random_state=1337,
        )

        model.fit(X_tr, y_tr)

        # Predict probabilities on the validation fold (choose threshold on validation)
        if hasattr(model, "predict_proba"):
            y_val_prob = model.predict_proba(X_val)[:, 1]
        else:
            # fallback (shouldn't happen for RF)
            y_val_prob = model.predict(X_val)

        thresholds = np.linspace(0.01, 0.99, 99)
        f1_scores = [f1_score(y_val, (y_val_prob > t).astype(int)) for t in thresholds]
        best_idx = int(np.nanargmax(f1_scores))
        best_threshold = thresholds[best_idx]

        y_val_pred = (y_val_prob > best_threshold).astype(int)
        f1_val = f1_score(y_val, y_val_pred)

        # report intermediate value to Optuna and allow pruning
        trial.report(f1_val, step=idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

        cv_scores[idx] = f1_val

    return float(np.mean(cv_scores))


In [ ]:
# Run study
start_time = time.time()

study = optuna.create_study(direction="maximize", study_name="Random Forest Charlie Hebdo",
                            sampler=optuna.samplers.TPESampler(seed=111113857),
                            pruner=optuna.pruners.MedianPruner())
func = lambda trial: objective_rf(trial, pd.DataFrame(X_train), y_train.astype(int))
study.optimize(func, n_trials=100)

end_time = time.time()

hyper_tuning_time = end_time - start_time

print(f"\tBest value (bcr1p_sum): {study.best_value:.5f}")
print(f"\tBest params:")

for key, value in study.best_params.items():
    print(f"\t\t{key}: {value}")


In [ ]:
best_params = study.best_params

In [8]:
best_params= {'n_estimators': 250,
 'max_depth': 4,
 'min_samples_split': 47,
 'min_samples_leaf': 64,
 'max_features': 'log2',
 'bootstrap': True,
 'class_weight': 'balanced_subsample'}

#### Example  training

In [10]:

model = RandomForestClassifier(
   **best_params,
    n_jobs=-1,
    random_state=42,
    verbose=False
)
# Train the model
model.fit(
    X_train,
    y_train
)


RandomForestClassifier(class_weight='balanced_subsample', max_depth=4,
                       max_features='log2', min_samples_leaf=64,
                       min_samples_split=47, n_estimators=250, n_jobs=-1,
                       random_state=42, verbose=False)

In [11]:
y_train_prob = model.predict_proba(X_train)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
print('Train Set: ')
evaluate(y_train, y_train_pred, y_train_prob, label="Train")
print('Test Set: ')
evaluate(y_test, y_test_pred, y_test_prob, label="Test")

Train Set: 
  - Accuracy:  0.8500
  - Precision: 0.8028
  - Recall:    0.9044
  - AUC:       0.9313

Test Set: 
  - Accuracy:  0.6733
  - Precision: 0.4929
  - Recall:    0.9286
  - AUC:       0.7977



#### Setting MLflow Experiment

In [12]:
mlflow.set_experiment("Random Forest 2025-11-04 Sydney Siege")

2025/11/09 17:32:42 INFO mlflow.tracking.fluent: Experiment with name 'Random Forest 2025-11-04 Sydney Siege' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/91', creation_time=1762709562225, experiment_id='91', last_update_time=1762709562225, lifecycle_stage='active', name='Random Forest 2025-11-04 Sydney Siege', tags={}>

#### Loading dataset statistics to get the final time cut 

In [13]:
df_posts_by_tm = pd.read_csv('sydneysiege_posts_by_time_cut.csv')

df_posts_by_tm['new_posts_cum_sum'] = df_posts_by_tm.new_posts.cumsum()

max_time_cut = int(df_posts_by_tm[df_posts_by_tm.new_posts_cum_sum==int(df_posts_by_tm.new_posts_cum_sum.max())]\
                   .time_cut.min())

In [ ]:
import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb

previous_node_count = 0

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\n=== Time Cut: {time_cut} ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = RandomForestClassifier(
               **best_params,
                n_jobs=-1,
                random_state=42
            )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]
 

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 40 ===
New Instances: 30

=== Time Cut: 70 ===
New Instances: 18

=== Time Cut: 100 ===
New Instances: 20

=== Time Cut: 130 ===
New Instances: 20

=== Time Cut: 160 ===
New Instances: 14

=== Time Cut: 190 ===
New Instances: 12

=== Time Cut: 220 ===
New Instances: 7

=== Time Cut: 250 ===
New Instances: 13

=== Time Cut: 280 ===
New Instances: 39

=== Time Cut: 310 ===
New Instances: 33

=== Time Cut: 340 ===
New Instances: 12

=== Time Cut: 370 ===
New Instances: 21

=== Time Cut: 400 ===
New Instances: 14

=== Time Cut: 430 ===
New Instances: 16

=== Time Cut: 460 ===
New Instances: 8

=== Time Cut: 490 ===
New Instances: 10

=== Time Cut: 520 ===
New Instances: 5

=== Time Cut: 550 ===
New Instances: 7

=== Time Cut: 580 ===
New Instances: 2

=== Time Cut: 610 ===
New Instances: 1

=== Time Cut: 640 ===
New Instances: 0

=== Time Cut: 670 ===
New Instances: 0

=== Time Cut: 700 ===
New Instances: 1

=== Time Cut: 730 ===
New Instances: 0

=== Time Cut: 760 ===
New I